In [0]:
from pyspark.sql import functions as F
from delta.tables import DeltaTable

In [0]:
%run /Workspace/Users/yvvswaraag@gmail.com/capgemini-data-engineering-training/FinalProject/1_setup/utilities


In [0]:
print(bronze_schema,silver_schema,gold_schema)

In [0]:
dbutils.widgets.text("catalog","bankingdataplatform","Catalog")
dbutils.widgets.text("data_source","batch","Data Source")

catalog = dbutils.widgets.get("catalog")
data_source = dbutils.widgets.get("data_source")

base_path = f"s3://swaraag-finalpro/Bankingdata/{data_source}"
print(base_path)


In [0]:
# define the tables
bronze_table = f"{catalog}.{bronze_schema}.{data_source}"
silver_table = f"{catalog}.{silver_schema}.{data_source}"
gold_table = f"{catalog}.{gold_schema}.{data_source}"

In [0]:
df = spark.read.options(header = True,inferSchema = True).csv(F"{base_path}/*.csv").withColumn("read_timestamp",F.current_timestamp()).select("*","_metadata.file_name","_metadata.file_size")
print("Total Rows: ", df.count())
df.show(5)

In [0]:
df.write.format("delta")\
    .option("enableChandeDataFeed","true").mode("append")\
        .saveAsTable(bronze_table)

%md
### Staging table to process just the arrived incremenal data

In [0]:
df.write\
 .format("delta") \
 .option("delta.enableChangeDataFeed", "true") \
 .mode("overwrite") \
 .saveAsTable(f"{catalog}.{bronze_schema}.staging_{data_source}")

## silver

In [0]:
from pyspark.sql.functions import col, upper, trim, when, current_timestamp

print("🔄 Starting Silver Layer Transformations...\n")

# Read from Bronze table
print(f"📖 Reading from Bronze: {bronze_table}")
bronze_df = spark.table(bronze_table)

initial_count = bronze_df.count()
print(f"   Initial records: {initial_count:,}\n")

# ============================================
# TRANSFORMATION 1: Deduplication by txn_id
# ============================================
print("✅ Rule 1: Deduplication by txn_id")
duplicates_count = bronze_df.count() - bronze_df.select("txn_id").distinct().count()
print(f"   Duplicates found: {duplicates_count:,}")

# Keep first occurrence, drop duplicates
silver_df = bronze_df.dropDuplicates(["txn_id"])
print(f"   After dedup: {silver_df.count():,} records\n")

# ============================================
# TRANSFORMATION 2: Validation - amount > 0
# ============================================
print("✅ Rule 2: Validation - Remove negative/zero amounts")
invalid_amounts = silver_df.filter((col("amount") <= 0) | col("amount").isNull()).count()
print(f"   Invalid amounts found: {invalid_amounts:,}")

silver_df = silver_df.filter(col("amount") > 0)
print(f"   After validation: {silver_df.count():,} records\n")

# ============================================
# TRANSFORMATION 3: Filtering - status = SUCCESS
# ============================================
print("✅ Rule 3: Filtering - Keep only SUCCESS transactions")
failed_pending = silver_df.filter(col("status") != "SUCCESS").count()
print(f"   FAILED/PENDING transactions: {failed_pending:,}")

silver_df = silver_df.filter(col("status") == "SUCCESS")
print(f"   After filtering: {silver_df.count():,} records\n")

# ============================================
# TRANSFORMATION 4: Normalization - txn_type to DEBIT/CREDIT only
# ============================================
print("✅ Rule 4: Normalization - Standardize txn_type to DEBIT/CREDIT")
print("   Original txn_type values:")
silver_df.groupBy("txn_type").count().orderBy("txn_type").show(20, truncate=False)

# Normalize txn_type: Map all variations to DEBIT or CREDIT
silver_df = silver_df.withColumn(
    "txn_type",
    when(
        upper(trim(col("txn_type"))).isin(["DEBIT", "DEBIT", "DR", "D", "DB"]),
        "DEBIT"
    ).when(
        upper(trim(col("txn_type"))).isin(["CREDIT", "CREDIT", "CR", "C"]),
        "CREDIT"
    ).otherwise("UNKNOWN")  # For any unexpected values
)

# Remove any UNKNOWN values (data quality)
silver_df = silver_df.filter(col("txn_type") != "UNKNOWN")

print("   After normalization:")
silver_df.groupBy("txn_type").count().orderBy("txn_type").show()

# ============================================
# Additional Data Quality: Handle NULLs
# ============================================
print("✅ Additional: Remove records with NULL critical fields")
null_check = silver_df.filter(
    col("account_id").isNull() | 
    col("customer_id").isNull() | 
    col("currency").isNull() |
    col("channel").isNull()
).count()
print(f"   Records with NULLs in critical fields: {null_check:,}")

silver_df = silver_df.filter(
    col("account_id").isNotNull() & 
    col("customer_id").isNotNull() & 
    col("currency").isNotNull() &
    col("channel").isNotNull()
)

print(f"   After NULL removal: {silver_df.count():,} records\n")

# ============================================
# Add Processing Metadata
# ============================================
silver_df = silver_df.withColumn("_processed_ts", current_timestamp())

# ============================================
# Final Summary
# ============================================
final_count = silver_df.count()
removed_count = initial_count - final_count
removal_pct = (removed_count / initial_count * 100) if initial_count > 0 else 0

print("="*70)
print("📊 SILVER LAYER TRANSFORMATION SUMMARY")
print("="*70)
print(f"Initial Bronze records:        {initial_count:,}")
print(f"Final Silver records:          {final_count:,}")
print(f"Records removed:               {removed_count:,} ({removal_pct:.1f}%)")
print(f"\nData Quality Issues Cleaned:")
print(f"  - Duplicates:                {duplicates_count:,}")
print(f"  - Invalid amounts:           {invalid_amounts:,}")
print(f"  - Failed/Pending:            {failed_pending:,}")
print(f"  - NULL values:               {null_check:,}")
print("="*70)

# Preview cleaned data
print("\n📋 Sample Silver Data (First 10 rows):")
display(silver_df.select(
    "txn_id", "account_id", "txn_time", "txn_type", 
    "amount", "currency", "channel", "status"
).limit(10))

In [0]:
# Write transformed data to Silver staging table
print(f"💾 Writing to Silver staging: {catalog}.{silver_schema}.staging_{data_source}")

silver_df.write \
    .format("delta") \
    .option("delta.enableChangeDataFeed", "true") \
    .mode("overwrite") \
    .saveAsTable(f"{catalog}.{silver_schema}.staging_{data_source}")

print(f"✅ Successfully written {silver_df.count():,} records to Silver staging table")
print(f"   Table: {catalog}.{silver_schema}.staging_{data_source}")

In [0]:
# ============================================
# GOLD LAYER: Incremental Load with MERGE
# ============================================

print("🏆 Starting Gold Layer Incremental Load\n")

# Read from Silver staging table
print(f"📖 Reading from Silver staging: {catalog}.{silver_schema}.staging_{data_source}")
silver_source = spark.table(f"{catalog}.{silver_schema}.staging_{data_source}")

record_count = silver_source.count()
print(f"📊 Records to process: {record_count:,}\n")

# Check if Gold table exists
if not spark.catalog.tableExists(gold_table):
    # ============================================
    # TABLE DOES NOT EXIST: Create new table
    # ============================================
    print(f"📝 Gold table does not exist. Creating: {gold_table}")
    
    silver_source.write \
        .format("delta") \
        .mode("overwrite") \
        .option("overwriteSchema", "true") \
        .option("delta.enableChangeDataFeed", "true") \
        .saveAsTable(gold_table)
    
    print(f"✅ Gold table created with {record_count:,} records")
    
else:
    
    print(f"🔄 Gold table exists. Performing incremental MERGE into: {gold_table}\n")
    
    # Get Delta table reference
    gold_delta = DeltaTable.forName(spark, gold_table)
    
    # Perform MERGE using txn_id as the unique key
    gold_delta.alias("target").merge(
        silver_source.alias("source"),
        "target.txn_id = source.txn_id"  # Match on transaction ID
    ).whenMatchedUpdateAll(  # Update if txn_id already exists
    ).whenNotMatchedInsertAll(  # Insert if txn_id is new
    ).execute()
    
    print("✅ MERGE completed successfully")
    print(f"   - Updated existing records with matching txn_id")
    print(f"   - Inserted new records")

# Verify final count
final_count = spark.table(gold_table).count()
print(f"\n🔍 Verification: {final_count:,} total records in {gold_table}")

